# 🌊 Negev Flood Detector — YAMNet Embeddings Training
Uses Google's pretrained YAMNet model to extract audio embeddings,
then trains a small classifier on top. Much faster and more accurate
than training a CNN from scratch.

**Pipeline:**
```
MP3/WAV → YAMNet → 1024-dim embedding → Dense classifier → TFLite
```

**Classes:** `flood_water` | `rain` | `ambient_dry`

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

# Load YAMNet and extract ONLY the embedding part
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

# Create a simple wrapper model that outputs embeddings
@tf.function(input_signature=[tf.TensorSpec(shape=[None], dtype=tf.float32)])
def get_embeddings(waveform):
    _, embeddings, _ = yamnet(waveform)
    return embeddings

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_concrete_functions(
    [get_embeddings.get_concrete_function()],
    yamnet
)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter.allow_custom_ops = True
tflite_model = converter.convert()

# Save it
with open('/content/yamnet_embeddings.tflite', 'wb') as f:
    f.write(tflite_model)

# Verify outputs
interp = tf.lite.Interpreter('/content/yamnet_embeddings.tflite')
interp.allocate_tensors()
print('Outputs:')
for d in interp.get_output_details():
    print(f'  index={d["index"]} shape={d["shape"]} name={d["name"]}')

print(f'\nSize: {len(tflite_model)/1024:.1f} KB')
print('✅ Done — download yamnet_embeddings.tflite from Colab files panel')

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import numpy as np

yamnet = hub.load('https://tfhub.dev/google/yamnet/1')

SAMPLES = 48000  # 3 seconds at 16kHz

# Build a concrete Keras model with fixed input shape
inputs = tf.keras.Input(shape=(SAMPLES,), dtype=tf.float32, name='waveform')

outputs = tf.keras.layers.Lambda(
    lambda x: tf.reduce_mean(
        yamnet(tf.squeeze(x, axis=0))[1],  # [1] = embeddings
        axis=0, keepdims=True
    )
)(inputs)

fixed_model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Test it works
test = np.random.randn(1, SAMPLES).astype(np.float32)
result = fixed_model(test)
print('Output shape:', result.shape)  # should be (1, 1024)

# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(fixed_model)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter.allow_custom_ops = True
tflite_model = converter.convert()

with open('/content/yamnet_fixed.tflite', 'wb') as f:
    f.write(tflite_model)

# Verify input shape
interp = tf.lite.Interpreter('/content/yamnet_fixed.tflite')
interp.allocate_tensors()
inp = interp.get_input_details()
out = interp.get_output_details()
print('Input shape:', inp[0]['shape'])   # must be [1, 48000]
print('Output shape:', out[0]['shape'])  # must be [1, 1024]
print('✅ Done')

## Step 1 — Mount Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tensorflow tensorflow-hub librosa
!apt-get install -q ffmpeg

print('✅ Ready')

## Step 2 — Imports & Config

In [ ]:
import os, glob, json
import numpy as np
import librosa
import tensorflow as tf
import tensorflow_hub as hub
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.utils import to_categorical

# ── Paths — same as your other notebooks ────────────────────
BASE_DIR   = '/content/drive/MyDrive/Python course/DIY/final_project'
DATA_DIR   = f'{BASE_DIR}/raw_audio'
OUTPUT_DIR = f'{BASE_DIR}/model'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSES     = ['flood_water', 'rain', 'ambient_dry']
SAMPLE_RATE = 16000   # YAMNet requires exactly 16kHz

print(f'✅ Config ready')
print(f'   Classes:     {CLASSES}')
print(f'   Sample rate: {SAMPLE_RATE} Hz (required by YAMNet)')

## Step 3 — Load YAMNet from TensorFlow Hub
YAMNet was trained by Google on 2 million YouTube clips across 521 audio classes.
We use it as a feature extractor — no retraining needed.

In [ ]:
print('📥 Loading YAMNet from TensorFlow Hub...')
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print('✅ YAMNet loaded')

# Quick test — verify it works
test_audio = np.zeros(SAMPLE_RATE, dtype=np.float32)  # 1 second of silence
scores, embeddings, spectrogram = yamnet_model(test_audio)
print(f'   Embedding shape per frame: {embeddings.shape}')
print(f'   → Each audio clip → multiple 1024-dim vectors (one per 0.48s frame)')

## Step 4 — Check Dataset

In [ ]:
print('📊 Dataset on Drive:')
print('=' * 45)
total = 0
for cls in CLASSES:
    files = glob.glob(f'{DATA_DIR}/{cls}/*.mp3') + glob.glob(f'{DATA_DIR}/{cls}/*.wav')
    total += len(files)
    bar = '█' * (len(files) // 5)
    print(f'  {cls:<20} {len(files):>4} files  {bar}')
print('=' * 45)
print(f'  TOTAL: {total} files')

## Step 5 — Extract YAMNet Embeddings
For each audio file:
1. Load and resample to 16kHz (YAMNet requirement)
2. Pass through YAMNet → get embedding frames
3. Average all frames → one 1024-dim vector per clip

This step replaces ALL of your mel spectrogram + CNN preprocessing.

In [ ]:
def extract_embedding(filepath):
    """
    For long files — extract multiple embeddings from different parts.
    Returns list of embeddings instead of one.
    """
    try:
        # Get file duration first
        duration = librosa.get_duration(path=filepath)

        # For short files — just load normally
        if duration <= 30:
            audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
            audio = audio / (np.max(np.abs(audio)) + 1e-6)
            _, embeddings, _ = yamnet_model(audio.astype(np.float32))
            return [np.mean(embeddings.numpy(), axis=0)]

        # For long files — sample 4 chunks of 10s from different positions
        n_samples = 4
        positions = np.linspace(0, duration - 10, n_samples)
        results = []

        for pos in positions:
            chunk, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True,
                                    offset=pos, duration=10)
            if len(chunk) < SAMPLE_RATE:  # skip too-short chunks
                continue
            chunk = chunk / (np.max(np.abs(chunk)) + 1e-6)
            _, embeddings, _ = yamnet_model(chunk.astype(np.float32))
            results.append(np.mean(embeddings.numpy(), axis=0))

        return results if results else [np.zeros(1024, dtype=np.float32)]

    except Exception as e:
        raise e

all_embeddings = []
all_labels     = []
skipped        = 0

for i_class, class_name in enumerate(CLASSES):
    files = glob.glob(f'{DATA_DIR}/{class_name}/*.mp3') + \
            glob.glob(f'{DATA_DIR}/{class_name}/*.wav')
    print(f'\n  [{class_name}] — {len(files)} files')

    for fpath in files:
        try:
            embs = extract_embedding(fpath)  # now returns a list
            for emb in embs:
                all_embeddings.append(emb)
                all_labels.append(i_class)
        except Exception as e:
            print(f'  ❌ FAILED: {os.path.basename(fpath)} — {e}')
            skipped += 1

    print(f'  ✅ {np.sum(np.array(all_labels) == i_class)} embeddings from [{class_name}]')

all_embeddings = np.array(all_embeddings, dtype=np.float32)
all_labels     = np.array(all_labels,     dtype=np.int32)

print(f'\n✅ Done!')
print(f'   Embeddings shape: {all_embeddings.shape}')
print(f'   Skipped: {skipped} files')

## Step 5b — Load Saved Embeddings (skip Step 5 on re-runs)
If you already ran Step 5, just run this cell instead to save time.

In [ ]:
# Uncomment these lines if you want to reload saved embeddings:
# all_embeddings = np.load(f'{OUTPUT_DIR}/embeddings.npy')
# all_labels     = np.load(f'{OUTPUT_DIR}/labels.npy')
# print(f'✅ Loaded embeddings: {all_embeddings.shape}')

## Step 6 — Augment Embeddings (balance classes)

In [ ]:
def augment_embedding(emb, n=1):
    """Add small Gaussian noise to embedding — simple but effective."""
    augmented = []
    for _ in range(n):
        noise = np.random.normal(0, 0.02, emb.shape).astype(np.float32)
        augmented.append(np.clip(emb + noise, -1, 1))
    return augmented


# Count per class before augmentation
counts = [np.sum(all_labels == i) for i in range(len(CLASSES))]
max_count = max(counts)

aug_embeddings = list(all_embeddings)
aug_labels     = list(all_labels)

for i_class, class_name in enumerate(CLASSES):
    class_count = counts[i_class]
    # How many times to augment to roughly balance classes
    factor = max(1, round(max_count / class_count) - 1)

    if factor > 0:
        idxs = np.where(all_labels == i_class)[0]
        for idx in idxs:
            for aug in augment_embedding(all_embeddings[idx], n=factor):
                aug_embeddings.append(aug)
                aug_labels.append(i_class)

all_embeddings = np.array(aug_embeddings, dtype=np.float32)
all_labels     = np.array(aug_labels,     dtype=np.int32)

print('📊 After augmentation:')
for i, cls in enumerate(CLASSES):
    count = np.sum(all_labels == i)
    bar   = '█' * (count // 10)
    print(f'  {cls:<20} {count:>5}  {bar}')
print(f'  TOTAL: {len(all_embeddings)}')

## Step 7 — Train / Test Split

In [ ]:
labels_cat = to_categorical(all_labels, num_classes=len(CLASSES))

X_train, X_test, Y_train, Y_test = train_test_split(
    all_embeddings, labels_cat,
    test_size=0.2, random_state=42,
    stratify=all_labels
)

print(f'✅ Split done')
print(f'   Train: {X_train.shape[0]} samples  shape: {X_train.shape}')
print(f'   Test:  {X_test.shape[0]} samples')

## Step 8 — Build Classifier on Top of YAMNet Embeddings
Just 3 Dense layers — input is already a rich 1024-dim feature vector from YAMNet.

In [ ]:
model = tf.keras.models.Sequential([
    # Input: 1024-dim YAMNet embedding
    tf.keras.layers.Input(shape=(1024,)),

    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),

    # Output: 3 classes
    tf.keras.layers.Dense(len(CLASSES), activation='softmax'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()
print(f'\nParameters: {model.count_params():,}  ← tiny compared to CNN from scratch')

## Step 9 — Train

In [ ]:
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    f'{OUTPUT_DIR}/best_yamnet_classifier.keras',
    monitor='val_loss', save_best_only=True, verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5,
    patience=5, min_lr=1e-6, verbose=1
)

history = model.fit(
    X_train, Y_train,
    epochs=100,          # more epochs ok — model is tiny, trains in seconds
    batch_size=32,
    validation_data=(X_test, Y_test),
    callbacks=[early_stop, checkpoint, reduce_lr]
)

with open(f'{OUTPUT_DIR}/yamnet_history.json', 'w') as f:
    json.dump(history.history, f)

print('\n✅ Training complete')

## Step 10 — Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     label='Train Loss',     color='red')
axes[0].plot(history.history['val_loss'], label='Val Loss',       color='blue')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['accuracy'],     label='Train Accuracy', color='red')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy',   color='blue')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle('YAMNet Flood Detector — Training')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/yamnet_training_curves.png', dpi=150)
plt.show()

## Step 11 — Evaluate & Confusion Matrix

In [ ]:
train_loss, train_acc = model.evaluate(X_train, Y_train, verbose=0)
val_loss,   val_acc   = model.evaluate(X_test,  Y_test,  verbose=0)

print(f'  Train accuracy: {train_acc:.3f}   loss: {train_loss:.3f}')
print(f'  Val   accuracy: {val_acc:.3f}   loss: {val_loss:.3f}')

y_pred     = model.predict(X_test)
y_pred_cls = np.argmax(y_pred,  axis=1)
y_true_cls = np.argmax(Y_test,  axis=1)

print('\n' + classification_report(y_true_cls, y_pred_cls, target_names=CLASSES))

cm = confusion_matrix(y_true_cls, y_pred_cls)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('YAMNet Flood Detector — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/yamnet_confusion_matrix.png', dpi=150)
plt.show()

## Step 12 — Build & Export Full Pipeline as TFLite
Combines YAMNet + your classifier into one single TFLite file.
The UNIHIKER just feeds raw audio in → gets class prediction out.

In [ ]:
# ── Export classifier-only TFLite (simplest, works best) ────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

classifier_path = f'{OUTPUT_DIR}/flood_classifier_only.tflite'
with open(classifier_path, 'wb') as f:
    f.write(tflite_model)

print(f'✅ Classifier TFLite saved → {classifier_path}')
print(f'   Size: {os.path.getsize(classifier_path)/1024:.1f} KB')

# ── Save class labels so UNIHIKER knows the order ────────────
labels_path = f'{OUTPUT_DIR}/class_labels.json'
with open(labels_path, 'w') as f:
    json.dump(CLASSES, f)
print(f'✅ Labels saved → {labels_path}')

print(f'\n📦 Files to copy to UNIHIKER:')
print(f'   1. {classifier_path}')
print(f'   2. {labels_path}')
print(f'\nOn UNIHIKER: YAMNet extracts embedding → classifier predicts class')

## Step 13 — Test Inference on a File

In [ ]:
import random
from IPython.display import Audio, display

def predict_file_yamnet(filepath):
    """Extract YAMNet embedding and predict class."""
    audio, _ = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    audio = (audio / (np.max(np.abs(audio)) + 1e-6)).astype(np.float32)

    _, embeddings, _ = yamnet_model(audio)
    mean_emb = np.mean(embeddings.numpy(), axis=0, keepdims=True)

    probs    = model.predict(mean_emb, verbose=0)[0]
    pred_cls = CLASSES[np.argmax(probs)]

    print(f'  File: {os.path.basename(filepath)}')
    for cls, prob in zip(CLASSES, probs):
        bar    = '█' * int(prob * 30)
        marker = ' ← predicted' if cls == pred_cls else ''
        print(f'  {cls:<20} {prob:>6.1%}  {bar}{marker}')
    print()
    return pred_cls


# Test one random file from each class
print('🧪 Inference test — one random file per class:\n')
for cls in CLASSES:
    files = glob.glob(f'{DATA_DIR}/{cls}/*.mp3') + glob.glob(f'{DATA_DIR}/{cls}/*.wav')
    if files:
        f = random.choice(files)
        print(f'  [{cls}]')
        display(Audio(f))
        predict_file_yamnet(f)

## ✅ Files Saved to Drive
```
MyDrive/.../model/
├── embeddings.npy                  ← cached YAMNet embeddings (skip re-extraction)
├── labels.npy
├── best_yamnet_classifier.keras    ← trained classifier
├── flood_detector_yamnet.tflite    ← full pipeline (YAMNet + classifier)
├── flood_classifier_only.tflite    ← classifier only (tiny, use with yamnet on UNIHIKER)
├── yamnet_training_curves.png
└── yamnet_confusion_matrix.png
```

## 🚀 UNIHIKER Deployment
On UNIHIKER use `flood_classifier_only.tflite` + install YAMNet separately:
```python
# On UNIHIKER
import tensorflow_hub as hub
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
# Record mic → yamnet → embedding → classifier → prediction
```